In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from gdt.core.data_primitives import TimeBins
from bctools.analysis import BayesianBlocksLightcurve
import os


grb_name = "bn240810880"
background_dir = Path("/data/background/dc4")
base_path = "/home/cosi/cosi/data/grb_eliza/test_t90"
#base_path = "/data/grb_eliza/sources/batch-1/flux-1-10/source_files/bn130307126/"
ori_path = "/home/cosi/cosi/data/background/dc4/DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"
start_ori_time = 1835487300.0
bin_size = 0.05


In [ ]:
def calculate_hist_lc(data_loaded,bin_size,g_tmin,g_tmax):

    bgo_z1 = data_loaded['z1']
    bgo_z0 = data_loaded['z0']
    bgo_x1 = data_loaded['x1']
    bgo_x0 = data_loaded['x0']
    bgo_y1 = data_loaded['y1']
    bgo_y0 = data_loaded['y0']

    data_array = [bgo_z1,bgo_z0,bgo_x1,bgo_x0,bgo_y1,bgo_y0]
    
    print("len data"+str(len(data_loaded)))

    bin_edges = np.arange(g_tmin, g_tmax + bin_size, bin_size)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    light_curve = np.zeros((len(bin_centers), 6, 2))

    for j, arr in enumerate(data_array):
        print("for")
        # 🔹 filtro sui tempi
        mask = (arr > g_tmin) & (arr < g_tmax)
        arr_filtered = arr[mask]
        frequenze, _ = np.histogram(arr_filtered, bins=bin_edges)
    
        light_curve[:, j, 0] = bin_edges[:-1]
        light_curve[:, j, 1] = frequenze

    return light_curve
    

In [ ]:
with open(base_path+"/"+grb_name+".time", "r") as file:
    contenuto = file.read().strip()   # rimuove spazi e newline
    grb_start_time = float(contenuto)

print(grb_start_time)
real_grb_start_time = start_ori_time+grb_start_time
print(real_grb_start_time)

with open(base_path+"/"+grb_name+".duration", "r") as file:
    contenuto = file.read().strip()   # rimuove spazi e newline
    duration = float(contenuto)

print(duration)


In [ ]:
def open_and_read_csv_counts(file,mass_model):
    
    col_names = [
        "Type",              
        "unix_time",         
        "x1", "x2", "x3",    
        "x4", "x5", "x6","x7",
        "SAA"            
    ]
    
    ori_df = pd.read_csv(
            ori_path,
            sep=r"\s+",         
            names=col_names,
            usecols=["unix_time", "SAA"],
            skiprows=1,              
            skip_blank_lines=True    
        )

    ori_df.set_index('unix_time', inplace=True)

    df = pd.read_csv(file)
    
    df.set_index('timestamp[s]', inplace=True)
               
    df_tmp = ori_df.reindex(df.index,method="nearest")
    df["SAA"] = df_tmp["SAA"]
    df_cut = df[df["SAA"]!=0]
    
    
            
    if mass_model == "DC3":
        bgo_times = {}
        for col in df.columns:
            if col.startswith("bgo"):
                filtered_data = df_cut[df_cut[col] > 0][[col]]
                times_in_seconds = filtered_data.index.to_numpy(dtype=float)
                bgo_times[col] = np.array(times_in_seconds)
        extracted_data = {'z1':bgo_times['bgo_z1[keV]'],
                        'z0':bgo_times['bgo_z0[keV]'],
                            'x1':bgo_times['bgo_x1[keV]'],
                            'x0':bgo_times['bgo_x0[keV]'],
                            'y1':bgo_times['bgo_y1[keV]'],
                            'y0':bgo_times['bgo_y0[keV]']}
    else:
        bgo_times = {}
        for col in df.columns:
            if col.startswith("SCB"):
                filtered_data = df_cut[df_cut[col] > 0][[col]]
                times_in_seconds = filtered_data.index.to_numpy(dtype=float)
                bgo_times[col] = np.array(times_in_seconds)
        extracted_data = {'z1':bgo_times['SCB2-A1[keV]'],
                        'z0':bgo_times['SCB2-A0[keV]'],
                            'x1':bgo_times['SCB0-A1[keV]'],
                            'x0':bgo_times['SCB0-A0[keV]'],
                            'y1':bgo_times['SCB1-A1[keV]'],
                            'y0':bgo_times['SCB1-A0[keV]']}
    
    return extracted_data

In [ ]:
""" # background processing


albedo_neutrons_file = background_dir/"AlbedoNeutrons_BGOhit_Total.npz"
albedo_photons_file = background_dir/"AlbedoPhotons_BGOhit_Total.npz"
cosmic_photons_file = background_dir/"CosmicPhotons_BGOhit_Total.npz"
primary_alphas_file = background_dir/"PrimaryAlphas_BGOhit_Total.npz"
primary_protons_file = background_dir/"PrimaryProtons_BGOhit_Total.npz"
secondary_electrons_file = background_dir/"SecondaryElectrons_BGOhit_Total.npz"
secondary_positrons_file = background_dir/"SecondaryPositrons_BGOhit_Total.npz" 
primary_electrons_file = background_dir/"PrimaryElectrons_BGOhit_Total.npz" 
secondary_protons_file = background_dir/"SecondaryProtons_BGOhit_Total.npz" 
saa_protons_file_1 = background_dir/"SAAprotons_BGOhit_Total_part1.npz" 
saa_protons_file_2 = background_dir/"SAAprotons_BGOhit_Total_part2.npz" 
saa_protons_file_3 = background_dir/"SAAprotons_BGOhit_Total_part3.npz" 
saa_protons_file_4 = background_dir/"SAAprotons_BGOhit_Total_part4.npz" 

g_tmin= start_ori_time
g_tmax = g_tmin+86400

import numpy as np

files = [
    albedo_neutrons_file,
    albedo_photons_file,
    cosmic_photons_file,
    primary_alphas_file,
    primary_protons_file,
    secondary_electrons_file,
    secondary_positrons_file,
    primary_electrons_file,
    secondary_protons_file,
    saa_protons_file_1,
    saa_protons_file_2,
    saa_protons_file_3,
    saa_protons_file_4,
]

merged = {
    'z1': [],
    'z0': [],
    'x1': [],
    'x0': [],
    'y1': [],
    'y0': []
}

# Carico e filtro subito (più efficiente)
for file in files:
    data = np.load(file)
    
    for key in merged:
        arr = data[key]
        mask = (arr > g_tmin) & (arr < g_tmax)
        merged[key].append(arr[mask])

# Concatenazione finale
for key in merged:
    merged[key] = np.concatenate(merged[key])

np.savez(background_dir/"TOTAL_BGOhit_Total_cut.npz", **merged)

print("File totale con cut creato.") """

In [ ]:
background_merged_cut = np.load(background_dir/"TOTAL_BGOhit_Total_cut.npz")

In [ ]:
grb_data = open_and_read_csv_counts(base_path+"/"+grb_name+".evt","DC3")

In [ ]:
data_array = [grb_data['z1'],grb_data['z0'],grb_data['x1'],grb_data['x0'],grb_data['y1'],grb_data['y0']]
background_data_array = [background_merged_cut['z1'],background_merged_cut['z0'],background_merged_cut['x1'],background_merged_cut['x0'],background_merged_cut['y1'],background_merged_cut['y0']] 

In [ ]:
background_lc = calculate_hist_lc(background_merged_cut,bin_size,real_grb_start_time-10,real_grb_start_time+duration+10)
grb_lc = calculate_hist_lc(grb_data,bin_size,0,duration)

In [ ]:
grb_lc

In [ ]:
def add_lc_from_time(background_lc, grb_lc, Y):
    """
    background_lc: (N, 6, 2)  -> [:, :, 0] = edges_left, [:, :, 1] = counts
    grb_lc:        (M, 6, 2)  -> edges_left parte da 0, counts
    Y: tempo assoluto nel riferimento del background dove inizia il GRB
    """
    bg = np.asarray(background_lc)
    grb = np.asarray(grb_lc)

    t_bg = bg[:, 0, 0]          # edges_left del background
    out = bg.copy()

    # primo bin del background con edge_left >= Y
    i0 = int(np.searchsorted(t_bg, Y, side="left"))

    # quanti bin posso sommare senza uscire
    L = min(out.shape[0] - i0, grb.shape[0])
    if L <= 0:
        return out

    # sommo i counts (tutti i 6 detector)
    out[i0:i0+L, :, 1] += grb[:L, :, 1]

    return out

In [ ]:
total_lc = add_lc_from_time(background_lc, grb_lc, Y=real_grb_start_time)

In [ ]:

np.savez(base_path+"/"+grb_name+"light_curve.npz", light_curve=total_lc)

In [ ]:

plt.step(background_lc[:,0,0][:500], background_lc[:,0,1][:500])
plt.axhline(y=np.mean(background_lc[:,0,1][:500]),color="red")  # retta orizzontale a y = 3
plt.step(grb_lc[:,0,0]+real_grb_start_time, grb_lc[:,0,1])
plt.xlabel("Time (s)")
plt.ylabel("Counts")
plt.show()

In [ ]:
np.mean(background_lc[:,0,1])

In [ ]:
plt.step(total_lc[:,0,0], total_lc[:,0,1])

In [ ]:

def save_results(GRB_dir, GRBname, signal_range1, signal_range2, t90, t90_error1, t90_error2, S, S_peak, Mode):
    """
    Save the results of the light curve analysis.
    """

    if Mode == "original": mode = "_original"
    elif Mode == "megalib": mode = "_megalib"
    elif Mode == "correction": mode = "_correction"
    else: mode = ""

    file_results_GRB = os.path.join(GRB_dir, f"bb_results"+mode+".dat")

    # Get GBM T90, flux and direction
    log_file = [f for f in os.listdir(GRB_dir) if f.endswith('.log')][0]
    source_file = [f for f in os.listdir(GRB_dir) if f.endswith('.source')][0]
    with open(log_file, "r") as f:
        for line in f:
            if "INFO: T90:" in line:
                items = line.split()
                t90_gbm = float(items[items.index("T90:") + 1])
    with open(source_file, 'r') as f:
        for line in f:
            if '.Flux' in line:
                line = line.strip()
                flux = float(line.split()[-1])
            if '.Beam' in line:
                line = line.strip()
                theta, phi = float(line.split()[-2]), float(line.split()[-1])
    
    with open(file_results_GRB, 'w') as f:
        f.write("# GRB_name flux[ph/s/cm2] theta[deg] phi[deg] t90_gbm[s] t_start[s] t_stop[s] t90[s] t90_error_low[s] t90_error_up[s] S S_peak\n")
        f.write(f"{GRBname} {flux} {theta} {phi} {t90_gbm} {signal_range1} {signal_range2} {t90} {t90_error1} {t90_error2} {S} {S_peak}\n")

def save_figure(ax, axzoom, GRB_dir, GRBname, signal_range1, signal_range2, Mode):
    """
    Save the figure of the light curve analysis.
    """

    if Mode == "original": 
        mode = "_original"
        title = "(Original)"
    elif Mode == "megalib":
        mode = "_megalib"
        title = "(MEGAlib correction)"
    elif Mode == "correction":
        mode = "_correction"
        title = "(Correction matrix)"
    else:
        mode = ""
        title = ""

    ax.set_title(f"{GRBname} {title}")
    axzoom.set_title(f"{GRBname} {title}")
    axzoom.set_xlim(signal_range1-5, signal_range2+5)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Rate [Hz]")
    axzoom.set_xlabel("Time [s]")
    axzoom.set_ylabel("Rate [Hz]")
    ax.legend()
    axzoom.legend()
    ax.grid(alpha=0.5)
    axzoom.grid(alpha=0.5)
    fig = ax.figure
    figzoom = axzoom.figure
    fig.tight_layout()
    figzoom.tight_layout()

    fig.savefig(os.path.join(GRB_dir, "bb_results"+mode+".pdf"))
    figzoom.savefig(os.path.join(GRB_dir, "bb_results_zoom"+mode+".pdf"))

    plt.close(fig)
    plt.close(figzoom)


def analyze_lc(lightcurve, p0=0.05, isRate=False, panels=['z0', 'z1', 'x0', 'x1', 'y0', 'y1']):
    """
    Analyze the light curve data.
    Input:
        - lightcurve: data file with times and counts
        - p0: false alarm probability for the Bayesian Block algorithm
        - isRate: if True, the lightcurve files contains rates instead of counts
        - panels: list of detectors for which we have lightcurves
    """

    data = lightcurve  
    time = data[:,0,0] 
    time = time - time[0]

    signal = {}
    for i, panel in enumerate(panels):
        signal[panel] = data[:, i, 1]
        if isRate:
            signal[panel] = signal[panel] * bin_width

    
    # Construct light curve object
    bin_width = time[1] - time[0]
    bin_edges = np.zeros(len(time) + 1)
    bin_edges[1:-1] = (time[:-1] + time[1:]) / 2
    bin_edges[0] = time[0] - (time[1] - time[0]) / 2
    bin_edges[-1] = time[-1] + (time[-1] - time[-2]) / 2
    lo_edges, hi_edges = bin_edges[:-1], bin_edges[1:]
    exposure = np.full(len(time), bin_width)


    lc = {} # lc per panel
    for panel in panels:
        signal_panel = signal[panel]
        lc[panel] = TimeBins(signal_panel, lo_edges, hi_edges, exposure)
    #if len(panels) > 1: lc_psum = TimeBins.sum([lcs for lcs in lc.values()])
    #else: lc_psum = lc[panels[0]]
    lc_psum = lc[panels[0]]

    lc_sel = lc_psum
    
    fig, ax = plt.subplots(figsize=(10,4))

    ax.step(lc_sel.centroids, lc_sel.counts, where="mid")

    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Counts / bin")
    ax.set_title("Light curve")
    ax.grid(True, alpha=0.3)

    plt.show()

    # Apply Bayesian Blocks algorithm
    
    print("min counts:", lc_sel.counts.min())
    print("bins with zero counts:", np.sum(lc_sel.counts == 0))

    print("exposure min:", lc_sel.exposure.min())
    print("exposure unique:", np.unique(lc_sel.exposure))
    print("any NaN in time:", np.isnan(time).any())
    print("any NaN in data:", np.isnan(data).any())

    try: 
        bb_lc = BayesianBlocksLightcurve(lc_sel)
        
        bb_lc.compute_bayesian_blocks(p0=p0)
        
        signal_range = bb_lc.signal_range
        
        t90 = bb_lc.duration(quantile = .9)
        
        t90_error = bb_lc.duration_error(.9, nsamples = 100)
        
    except Exception as e:
        print(e)
        print('WARNING: ')
        return lc_sel, None, -9999, -9999, -9999, -9999, -9999, -9999, -9999, None, None
    fig,ax = plt.subplots()
    fig2,ax2 = plt.subplots()
    ax = bb_lc.plot(ax=ax)
    ax2 = bb_lc.plot(ax=ax2)

    # Li&Ma calculation of the significance
    signal_lc = lc_sel.slice(bb_lc.signal_range.tstart, bb_lc.signal_range.tstop)
    bkg_lc = lc_sel.slice(bb_lc.signal_range.tstop, lc_sel.centroids[-1])
    bkg_lc = lc_sel.slice(lc_sel.centroids[0], bb_lc.signal_range.tstart)
    t_on = np.sum(signal_lc.exposure)
    t_off = np.sum(bkg_lc.exposure)
    alpha = t_on / t_off
    N_on = np.sum(signal_lc.rates * signal_lc.exposure)
    N_off = np.sum(bkg_lc.rates * bkg_lc.exposure)
    S = np.sqrt(2) * ( N_on * np.log( ((1+alpha)/alpha) * (N_on/(N_on+N_off)) ) + N_off * np.log( (1+alpha) * (N_off/(N_on+N_off)) ) ) ** 0.5

    # Li&Ma calculation of the peak significance
    significance = []
    for rate, exp in zip(signal_lc.rates, signal_lc.exposure):
        N_on = rate * exp
        t_on = exp
        alpha = t_on / t_off
        S_bin = np.sqrt(2) * ( N_on * np.log( ((1+alpha)/alpha) * (N_on/(N_on+N_off)) ) + N_off * np.log( (1+alpha) * (N_off/(N_on+N_off)) ) ) ** 0.5
        significance.append(S_bin)
    significance = np.array(significance)
    S_peak = np.max(significance)

    result = (lc_sel, bb_lc, signal_range.tstart, signal_range.tstop, t90, t90_error[0], t90_error[1], S, S_peak, ax, ax2,lo_edges,hi_edges)
    plt.show
 
    return result


In [ ]:
output = analyze_lc(total_lc, p0=10e-5, isRate=False, panels=['z1', 'z0', 'x1', 'x0', 'y1', 'y0'])

In [ ]:
output[0]

In [ ]:

x = []
y = []

with open(base_path+"/"+grb_name+".dat", "r") as file:
    for row in file:
        if "#" in row or "EN" in row:
            continue
        component = row.strip().split()
        
        x.append(float(component[1]))
        y.append(float(component[2]))

# Plot
plt.plot(x, y)
plt.xlabel("X")
plt.ylabel("Y")
plt.title(".dat plot")
plt.grid()
plt.show()

In [ ]:
import numpy as np
from scipy.stats import norm, skewnorm, gennorm, johnsonsu,poisson
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# Funzione per calcolare il chi-quadro
def chi2(data, expected, bin_counts, bin_errors, n_parameters):
    mask = expected > 0
    chi2_val = np.sum((bin_counts[mask] - expected[mask]) ** 2 / expected[mask])
    ndof = len(bin_counts) - len(n_parameters)  # Gradi di libertà, calcolati come numero di bin - numero di parametri
    return chi2_val, ndof, chi2_val / ndof

# Funzione modello per Gaussiana
def gaussian(x, mu, sigma, amp):
    return amp * norm.pdf(x, loc=mu, scale=sigma)

# Funzione modello per Skew-Normal
def skewnormal(x, alpha, mu, sigma, amp):
    return amp * skewnorm.pdf(x, alpha, loc=mu, scale=sigma)

# Funzione modello per Generalized Normal
def gennormal(x, beta, mu, sigma, amp):
    return amp * gennorm.pdf(x, beta, loc=mu, scale=sigma)

# Funzione modello per Johnson SU
def johnsonsu_model(x, gamma, delta, mu, sigma, amp):
    return amp * johnsonsu.pdf(x, gamma, delta, loc=mu, scale=sigma)

# Calcola il numero di bin usando la regola di Sturges
def sturges_rule(data):
    n = len(data)
    return int(1 + np.log2(n))

# Funzione modello per Poisson
def poisson_model(x, lambd, amp):
    return amp * poisson.pmf(np.round(x), lambd)

import scipy.stats as stats

def f_test(chi2_1, dof_1, chi2_2, dof_2):
    """
    Esegue l'F-test per confrontare due valori di chi-quadro da due modelli di fit.
    
    Parametri:
    chi2_1: float - Chi-quadro del primo modello (deve essere maggiore di chi2_2)
    dof_1: int - Gradi di libertà del primo modello
    chi2_2: float - Chi-quadro del secondo modello
    dof_2: int - Gradi di libertà del secondo modello
    
    Ritorna:
    F: float - Statistica F
    p_value: float - p-value associato al test
    """
    if chi2_1 < chi2_2:
        chi2_1, chi2_2 = chi2_2, chi2_1
        dof_1, dof_2 = dof_2, dof_1
    
    df_numerator = dof_1 - dof_2
    df_denominator = dof_2
    
    if df_numerator <= 0:
        raise ValueError("I gradi di libertà devono soddisfare dof_1 > dof_2.")
    
    F = ((chi2_1 - chi2_2) / df_numerator) / (chi2_2 / df_denominator)
    p_value = 1 - stats.f.cdf(F, df_numerator, df_denominator)
    
    return F, p_value



def calculate_fit_and_plot(data,bins):

    # Calcolo degli histogrammi
    bin_counts, bin_edges = np.histogram(data, bins=bins, density=False)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_errors = np.sqrt(bin_counts)  # Errori Poissoniani
    
    # Parametri iniziali per il fit
    p0_gaussian = [np.mean(data), np.std(data), max(bin_counts)]
    p0_skewnormal = [0, np.mean(data), np.std(data), max(bin_counts)]
    p0_gennorm = [2, np.mean(data), np.std(data), max(bin_counts)]
    p0_johnsonsu = [0, 1, np.mean(data), np.std(data), max(bin_counts)]
    p0_poisson = [np.mean(data), max(bin_counts)]
    
    # Fit con le distribuzioni alternative
    popt_gaussian, _ = curve_fit(gaussian, bin_centers, bin_counts, sigma=bin_errors, p0=p0_gaussian, absolute_sigma=True,maxfev=50000)
    popt_skewnormal, _ = curve_fit(skewnormal, bin_centers, bin_counts, sigma=bin_errors, p0=p0_skewnormal, absolute_sigma=True,maxfev=50000)
    popt_gennorm, _ = curve_fit(gennormal, bin_centers, bin_counts, sigma=bin_errors, p0=p0_gennorm, absolute_sigma=True,maxfev=50000)
    popt_johnsonsu, _ = curve_fit(johnsonsu_model, bin_centers, bin_counts, sigma=bin_errors, p0=p0_johnsonsu, absolute_sigma=True,maxfev=50000)
    popt_poisson, _ = curve_fit(poisson_model, bin_centers, bin_counts, sigma=bin_errors, p0=p0_poisson, absolute_sigma=True,maxfev=50000)

    print(popt_gaussian)
    print(popt_skewnormal)
    print(popt_gennorm)
    print(popt_johnsonsu)
    print(popt_poisson)
    
    # Calcolo delle distribuzioni attese
    expected_gaussian = gaussian(bin_centers, *popt_gaussian)
    expected_skewnormal = skewnormal(bin_centers, *popt_skewnormal)
    expected_gennorm = gennormal(bin_centers, *popt_gennorm)
    expected_johnsonsu = johnsonsu_model(bin_centers, *popt_johnsonsu)
    expected_poisson = poisson_model(bin_centers, *popt_poisson)
    
    # Calcolo del chi-quadro per ogni modello
    chi2_gaussian, ndof_gaussian, chi2_r_gaussian = chi2(data, expected_gaussian, bin_counts, bin_errors, popt_gaussian)
    chi2_skewnormal, ndof_skewnormal, chi2_r_skewnormal = chi2(data, expected_skewnormal, bin_counts, bin_errors, popt_skewnormal)
    chi2_gennorm, ndof_gennorm, chi2_r_gennorm = chi2(data, expected_gennorm, bin_counts, bin_errors, popt_gennorm)
    chi2_johnsonsu, ndof_johnsonsu, chi2_r_johnsonsu = chi2(data, expected_johnsonsu, bin_counts, bin_errors, popt_johnsonsu)
    chi2_poisson, ndof_poisson, chi2_r_poisson = chi2(data, expected_poisson, bin_counts, bin_errors, popt_poisson)
    
    # Stampa dei risultati
    print(f"Chi2 (Gaussian): {chi2_gaussian:.2f}, Ndof: {ndof_gaussian}, Chi2/Ndof: {chi2_r_gaussian:.2f}")
    print(f"Chi2 (Skew-Normal): {chi2_skewnormal:.2f}, Ndof: {ndof_skewnormal}, Chi2/Ndof: {chi2_r_skewnormal:.2f}")
    print(f"Chi2 (Generalized Normal): {chi2_gennorm:.2f}, Ndof: {ndof_gennorm}, Chi2/Ndof: {chi2_r_gennorm:.2f}")
    print(f"Chi2 (Johnson SU): {chi2_johnsonsu:.2f}, Ndof: {ndof_johnsonsu}, Chi2/Ndof: {chi2_r_johnsonsu:.2f}")
    print(f"Chi2 (Poisson): {chi2_poisson:.2f}, Ndof: {ndof_poisson}, Chi2/Ndof: {chi2_r_poisson:.2f}")


    F_stat, p_val = f_test(chi2_gaussian, ndof_gaussian, chi2_skewnormal, ndof_skewnormal)
    print(f"F-statistic: {F_stat}, p-value: {p_val}")


    # Plot dei risultati
    plt.bar(bin_centers, bin_counts / len(data), width=bin_edges[1] - bin_edges[0], alpha=0.5, label="Data")
    plt.plot(bin_centers, expected_gaussian / len(data), label="Gaussian Fit", color='red')
    plt.plot(bin_centers, expected_skewnormal / len(data), label="Skew-Normal Fit", color='green')
    plt.plot(bin_centers, expected_gennorm / len(data), label="Generalized Normal Fit", color='blue')
    plt.plot(bin_centers, expected_johnsonsu / len(data), label="Johnson SU Fit", color='orange')
    plt.plot(bin_centers, expected_poisson / len(data), label="Poisson Fit", color='purple', linestyle='dashed')
    
    plt.errorbar(bin_centers, bin_counts / len(data), yerr=bin_errors / len(data), fmt=' ', color='blue', label="Errors", capsize=5)
    
    plt.legend()
    plt.xlabel("Panel's counts (bins of "+str(bin_size)+" second)")
    plt.ylabel("Normalized Frequency")
    plt.show()

In [ ]:
background_lc = background_lc[:500]

In [ ]:
data = background_lc[:,1,1]
calculate_fit_and_plot(data,21)

In [ ]:
print(np.mean(data))
print(np.std(data))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, poisson

# Parametri
mu = 38.99
sigma = 8.48


# ---------------------
# Gaussiana (continua)
# ---------------------
x = np.linspace(-100, 200, 1000)  # intervallo ampio per vedere bene la curva
gauss = norm.pdf(x, mu, sigma)

# ---------------------
# Poisson (discreta)
# ---------------------
k = np.arange(0, 150)  # valori interi
poiss = poisson.pmf(k, mu)

# ---------------------
# Plot
# ---------------------
plt.figure()

plt.plot(x, gauss, label='Gaussiana μ=50, σ=50')
plt.plot(k, poiss, label='Poisson λ=50')

plt.xlabel('x')
plt.ylabel('Densità / Probabilità')
plt.title('Confronto Gaussiana e Poisson')
plt.legend()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson, norm
from scipy.special import gammaln

# --- i tuoi dati ---
counts  = data

N = len(counts)

# =========================
# 1) FIT POISSON (MLE)
# =========================
lam_hat = counts.mean()

# log-likelihood Poisson (per controllo / confronto)
# log P(x|λ) = x log λ - λ - log(x!)
def poisson_nll(lam, x):
    if lam <= 0:
        return np.inf
    return -(x*np.log(lam) - lam - gammaln(x+1)).sum()

nll_pois = poisson_nll(lam_hat, counts)

# errore circa su λ (se Poisson i.i.d.)
lam_err = np.sqrt(lam_hat / N)

print(f"Poisson fit: lambda = {lam_hat:.4f} ± {lam_err:.4f}  (N={N})")
print(f"Poisson NLL @ lambda_hat: {nll_pois:.3f}")

# =========================
# 2) FIT GAUSSIANA (MLE)
# =========================
mu_hat, sigma_hat = norm.fit(counts)  # MLE per normale

# negative log-likelihood gaussiana
nll_gauss = -norm.logpdf(counts, loc=mu_hat, scale=sigma_hat).sum()

print(f"Gaussian fit: mu = {mu_hat:.4f}, sigma = {sigma_hat:.4f}")
print(f"Gaussian NLL @ (mu_hat,sigma_hat): {nll_gauss:.3f}")

# =========================
# 3) PLOT: istogramma + overlay
# =========================
kmin, kmax = counts.min(), counts.max()
k = np.arange(kmin, kmax + 1)  # support discreto

# istogramma con bin centrati sugli interi (IMPORTANTISSIMO per Poisson)
bins = np.arange(kmin - 0.5, kmax + 1.5, 1.0)

plt.figure()
plt.hist(counts, bins=bins, density=True, alpha=0.35, label="Data (hist, density)")

# Poisson: PMF sui k interi (linea a punti sui centri)
pmf = poisson.pmf(k, mu=lam_hat)
plt.plot(k, pmf, "o-", label=f"Poisson(λ={lam_hat:.2f})")

# Gaussiana: PDF continua
xgrid = np.linspace(kmin - 3, kmax + 3, 600)
pdf = norm.pdf(xgrid, loc=mu_hat, scale=sigma_hat)
plt.plot(xgrid, pdf, "-", label=f"Gaussian(μ={mu_hat:.2f}, σ={sigma_hat:.2f})")

plt.xlabel("Counts per 50 ms")
plt.ylabel("Probability density")
plt.legend()
plt.tight_layout()
plt.show()

# =========================
# 4) CONFRONTO NUMERICO (AIC, opzionale)
# =========================
# AIC = 2k + 2*NLL, dove k=numero parametri
AIC_pois = 2*1 + 2*nll_pois       # λ
AIC_gauss = 2*2 + 2*nll_gauss     # μ, σ
print(f"AIC Poisson:  {AIC_pois:.2f}")
print(f"AIC Gaussian: {AIC_gauss:.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson, norm, skewnorm, johnsonsu
from scipy.special import gammaln

# --- i tuoi dati ---
counts = data
counts = np.asarray(counts)

N = len(counts)

# =========================
# 1) FIT POISSON (MLE)
# =========================
lam_hat = counts.mean()

def poisson_nll(lam, x):
    if lam <= 0:
        return np.inf
    return -(x*np.log(lam) - lam - gammaln(x+1)).sum()

nll_pois = poisson_nll(lam_hat, counts)
lam_err = np.sqrt(lam_hat / N)

print(f"Poisson fit: lambda = {lam_hat:.4f} ± {lam_err:.4f}  (N={N})")
print(f"Poisson NLL @ lambda_hat: {nll_pois:.3f}")

# =========================
# 2) FIT GAUSSIANA (MLE)
# =========================
mu_hat, sigma_hat = norm.fit(counts)
nll_gauss = -norm.logpdf(counts, loc=mu_hat, scale=sigma_hat).sum()

print(f"Gaussian fit: mu = {mu_hat:.4f}, sigma = {sigma_hat:.4f}")
print(f"Gaussian NLL @ (mu_hat,sigma_hat): {nll_gauss:.3f}")

# =========================
# 3) FIT SKEW-NORMAL (MLE)
# =========================
# Parametri: a (shape), loc, scale
a_hat, loc_sn, scale_sn = skewnorm.fit(counts)
nll_skewnorm = -skewnorm.logpdf(counts, a_hat, loc=loc_sn, scale=scale_sn).sum()

print(f"Skew-normal fit: a = {a_hat:.4f}, loc = {loc_sn:.4f}, scale = {scale_sn:.4f}")
print(f"Skew-normal NLL: {nll_skewnorm:.3f}")

# =========================
# 4) FIT JOHNSON SU (MLE)
# =========================
# Parametri: a, b (shape), loc, scale
a_jsu, b_jsu, loc_jsu, scale_jsu = johnsonsu.fit(counts)
nll_jsu = -johnsonsu.logpdf(counts, a_jsu, b_jsu, loc=loc_jsu, scale=scale_jsu).sum()

print(f"Johnson SU fit: a = {a_jsu:.4f}, b = {b_jsu:.4f}, loc = {loc_jsu:.4f}, scale = {scale_jsu:.4f}")
print(f"Johnson SU NLL: {nll_jsu:.3f}")

# =========================
# 5) PLOT: istogramma + overlay
# =========================
kmin, kmax = counts.min(), counts.max()
k = np.arange(kmin, kmax + 1)
bins = np.arange(kmin - 0.5, kmax + 1.5, 1.0)

plt.figure()
plt.hist(counts, bins=bins, density=True, alpha=0.35, label="Data (hist, density)")

# Poisson (discreta)
pmf = poisson.pmf(k, mu=lam_hat)
plt.plot(k, pmf, "o-", label=f"Poisson(λ={lam_hat:.2f})")

# Griglia continua per le PDF
xgrid = np.linspace(kmin - 3, kmax + 3, 800)

# Gauss
pdf_gauss = norm.pdf(xgrid, loc=mu_hat, scale=sigma_hat)
plt.plot(xgrid, pdf_gauss, "-", label=f"Gaussian(μ={mu_hat:.2f}, σ={sigma_hat:.2f})")

# Skew-normal
pdf_sn = skewnorm.pdf(xgrid, a_hat, loc=loc_sn, scale=scale_sn)
plt.plot(xgrid, pdf_sn, "-", label=f"SkewNorm(a={a_hat:.2f})")

# Johnson SU
pdf_jsu = johnsonsu.pdf(xgrid, a_jsu, b_jsu, loc=loc_jsu, scale=scale_jsu)
plt.plot(xgrid, pdf_jsu, "-", label=f"JohnsonSU(a={a_jsu:.2f}, b={b_jsu:.2f})")

plt.xlabel("Counts per 50 ms")
plt.ylabel("Probability density")
plt.legend()
plt.tight_layout()
plt.show()

# =========================
# 6) CONFRONTO NUMERICO (AIC)
# =========================
# AIC = 2k + 2*NLL, k = numero parametri del modello
AIC_pois  = 2*1 + 2*nll_pois                 # λ
AIC_gauss = 2*2 + 2*nll_gauss                # μ, σ
AIC_sn    = 2*3 + 2*nll_skewnorm             # a, loc, scale
AIC_jsu   = 2*4 + 2*nll_jsu                  # a, b, loc, scale

print("\nAIC:")
print(f"  Poisson:     {AIC_pois:.2f}")
print(f"  Gaussian:    {AIC_gauss:.2f}")
print(f"  Skew-normal: {AIC_sn:.2f}")
print(f"  Johnson SU:  {AIC_jsu:.2f}")

print("\n=== PARAMETRI FIT ===")

params_pois = np.array([lam_hat])
params_gauss = np.array([mu_hat, sigma_hat])
params_skewnorm = np.array([a_hat, loc_sn, scale_sn])
params_jsu = np.array([a_jsu, b_jsu, loc_jsu, scale_jsu])

print("Poisson params =", params_pois)
print("Gaussian params =", params_gauss)
print("SkewNormal params =", params_skewnorm)
print("JohnsonSU params =", params_jsu)

In [ ]:
panel = 5
counts_bg = background_lc[:,panel,1]
t = background_lc[:,panel,0]-np.min(background_lc[:,panel,0])
# ===== fit polinomiale =====
order = 2   # puoi cambiare ordine
coeff = np.polyfit(t, counts_bg, order)

poly = np.poly1d(coeff)

# ===== stampa coefficienti =====
print("Coefficienti del polinomio:")
print(coeff)

print("\nPolinomio:")
print(poly)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))

plt.scatter(t, counts_bg, s=5, label="background simulato")
plt.plot(t, poly(t), color="red", label="fit polinomiale")

plt.xlabel("time (s)")
plt.ylabel("counts / bin")
plt.legend()

plt.show()

In [ ]:
residuals = counts_bg - poly(t)

plt.figure(figsize=(8,3))
plt.scatter(t, residuals, s=5)
plt.axhline(0,color="red")
plt.xlabel("time")
plt.ylabel("residuals")
plt.show()

In [ ]:
print("std residui:", np.std(residuals))
print("sqrt(mean counts):", np.sqrt(np.mean(counts_bg)))